# 04 — Quench fraction vs azimuthal angle (MW-mass hosts)

For Milky-Way-mass hosts (`12.0 < log10 M_200c < 12.5`, physical) and the seven half-dex
satellite stellar-mass cuts `log10 M*,sat = 6.0, 6.5, ..., 9.0`, plot the satellite **quenched
fraction as a function of azimuthal angle** from the host major axis, folded to **[0, 90]**
(0 = major axis, 90 = minor axis).

**Runs only on Binder** (reads raw TNG via `tng_utils.compute_satellite_catalog`). The little-h
convention, host selection, angle fold and quench definition all live in `tng_utils.py`, shared
with notebook 01 so they cannot drift apart.

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt

# make sure the shared module (sits next to this notebook) is importable
sys.path.insert(0, os.getcwd())
import tng_utils as T

## Configuration

In [ ]:
SIM      = 'TNG100'                                       # 'TNG100' or 'TNG50'
REDSHIFT = 'z0'                                           # 'z0' (snap 99) or 'z0p05' (snap 98)
TNG_ROOT = T.sim_root(SIM)                                # ~/SimulationData/<sim_dir> (edit in tng_utils if needed)
SNAP     = T.snap_for(REDSHIFT)

HOST_LOGM_MIN, HOST_LOGM_MAX = 12.0, 12.5                 # MW-mass hosts (physical M_sun)
SAT_LOGM_CUTS = np.array([6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0])   # log10 M*,sat lower limits

BINW = 5.0                                                # angle bin width [deg]
NBINS_90 = int(90 / BINW)

assert os.path.isdir(os.path.join(TNG_ROOT, 'output')), f'set TNG_ROOT for {SIM}: {TNG_ROOT} not found'
print(f'SIM = {SIM} | REDSHIFT = {REDSHIFT} (snap {SNAP})')

## Build the satellite catalog

One per-satellite table for all MW-mass hosts (no satellite-mass cut yet — we apply those below).
Columns include `alpha` (0-90 deg from major axis), `mstar_phys`, `quenched`.

In [ ]:
df = T.compute_satellite_catalog(TNG_ROOT, HOST_LOGM_MIN, HOST_LOGM_MAX, sim=SIM, snap=SNAP)
print(f'{SIM} {REDSHIFT} MW-mass hosts: {df.host_id.nunique()} | satellites (any mass): {len(df)}')
print(f'overall f_q (M* > 1e6): {df[df.mstar_phys > 6].quenched.mean():.3f}')

## Quench fraction vs angle, per satellite mass cut

Each curve is one lower mass limit (colored from low mass = blue to high mass = red); error bars
are binomial, `sqrt(f(1-f)/N)`.

In [ ]:
colors = plt.cm.coolwarm(np.linspace(0, 1, len(SAT_LOGM_CUTS)))

fig, ax = plt.subplots(figsize=(8, 6))
for logcut, col in zip(SAT_LOGM_CUTS, colors):
    sub = df[df.mstar_phys > logcut]
    c, fq, err, n = T.fq_vs_angle(sub.alpha, sub.quenched, NBINS_90, 90)
    ax.errorbar(c, fq, yerr=err, color=col, marker='o', ms=4, lw=1.5, capsize=2,
                label=f'{logcut:.1f}')

ax.set_xlim(0, 90); ax.set_ylim(0, 1)
ax.set_xlabel(r'$\theta$ from major axis [deg]')
ax.set_ylabel(r'quenched fraction $f_q$')
ax.tick_params(which='both', direction='in', top=True, right=True)
ax.legend(title=r'$\log_{10} M_{*,\rm sat}$ cut', fontsize=9, ncol=2, fancybox=False)
ax.set_title(rf'{SIM} ({REDSHIFT})  MW-mass hosts (${HOST_LOGM_MIN}<\log_{{10}}M_{{200c}}<{HOST_LOGM_MAX}$)')
plt.tight_layout(); plt.show()

## Notes

* Sample size drops quickly toward the high-mass cuts (`log10 M* = 8.5, 9.0`), so those error bars
  are large; for the smallest cut some low-mass satellites have `SubhaloSFR = 0` (TNG resolution
  floor) and count as quenched, lifting f_q at the low-mass end.
* To use h^-1 satellite masses instead, cut on `mstar_hinv` (the column is already in the table).